In [ ]:
import pandas as pd
import numpy as np
import math, os


from utils import PARAMS

In [ ]:
def analyze_fitts_trials(df, hold_required=120):

    df = df.reset_index(drop=True)  # FIX

    df["new_trial"] = (
        (df["target_x"].shift() != df["target_x"]) |
        (df["target_y"].shift() != df["target_y"])
    )

    trial_indices = df.index[df["new_trial"]].tolist()
    if 0 not in trial_indices:
        trial_indices.insert(0, 0)

    results = []

    for i in range(len(trial_indices)):
        start_idx = trial_indices[i]
        end_idx = trial_indices[i + 1] - 1 if i + 1 < len(trial_indices) else len(df) - 1

        trial = df.iloc[start_idx:end_idx + 1]
        if trial.empty:
            continue

        t_start = trial["time"].iloc[0]
        t_end = trial["time"].iloc[-1]
        MT = t_end - t_start

        start_x, start_y = trial["cursor_x"].iloc[0], trial["cursor_y"].iloc[0]
        target_x, target_y = trial["target_x"].iloc[0], trial["target_y"].iloc[0]
        W = trial["radius"].iloc[0] * 2

        D = math.hypot(target_x - start_x, target_y - start_y)
        ID = math.log2(D / W + 1) if W > 0 else np.nan
        TP = ID / MT if MT > 0 else np.nan

        success = int(trial["hold_count"].max() >= hold_required - 1)

        dx = np.diff(trial["cursor_x"].values)
        dy = np.diff(trial["cursor_y"].values)
        path_length = np.sum(np.sqrt(dx**2 + dy**2))
        path_eff = D / path_length if path_length > 0 else np.nan

        inside = trial["inside"].astype(bool).values
        crossings = np.sum(inside[1:] != inside[:-1]) // 2

        results.append({
            "Trial": i + 1,
            "MT": MT,
            # "D": D,
            # "W": W,
            # "ID": ID,
            "TP": TP,
            "Win": success,
            "PE": path_eff,
            "Cross": crossings
        })

    return pd.DataFrame(results)


In [ ]:
NAME = 'A'
path = f'fitts_logs/{NAME}/'

files =  sorted(os.listdir(path))

for f in files:
    print('\n\n\n',f)
    df = pd.read_csv(path + f)
    trial_metrics = analyze_fitts_trials(df, hold_required=PARAMS['hold_frames_required'])
    trial_metrics.describe().to_csv("da_p.csv")
    trial_metrics.to_csv("a_p.csv")
    print(trial_metrics)
    print(trial_metrics.describe())




 Fitts_cnn_raw_2026-02-17_02-55-57.csv
    Trial        MT        TP  Win        PE  Cross
0       1  2.108749  1.105492    1  0.987167      0
1       2  2.608693  1.220077    1  1.048676      1
2       3  3.423543  0.913715    1  0.605520      1
3       4  4.351598  0.725622    1  0.488572      2
4       5  4.255479  0.752139    1  0.448004      1
5       6  5.264631  0.615455    1  0.491744      2
6       7  5.040062  0.623739    1  0.501747      4
7       8  2.799805  1.107387    1  0.782301      1
8       9  6.480470  0.480011    1  0.266979      3
9      10  2.799690  1.139143    1  1.050175      1
10     11  4.096213  0.764367    1  0.515505      2
11     12  4.064586  0.778880    1  0.588240      1
12     13  3.999595  0.807897    1  0.486076      3
13     14  4.688101  0.674485    1  0.369744      2
14     15  4.640714  0.677462    1  0.576545      3
15     16  3.520054  0.914651    1  0.589224      1
           Trial         MT         TP   Win         PE      Cross
count  